# HLS Video Player — Google Colab 起動ノート

このノートブックは hls-video-player を Colab 上で動作させます。**新アーキテクチャ（フォルダ走査モデル）**:

1. **アプリケーション本体** は GitHub から `git clone` で取得 (Drive 配置不要)
2. **動画フォルダ** は Drive 上の任意の場所を **GUI で指定**
3. **CLI で一括変換** — 指定フォルダ直下に `converted/{stem}/{hls,thumbs,meta.json}` が生成される
4. **GUI（閲覧専用）** — 設定された動画フォルダを走査して、ホバーサムネ付きで一覧・再生

## 事前準備（初回のみ）

### A. アプリケーションの GitHub リポジトリ

https://github.com/motowo/hls-video-player を `git clone` で取得します（Drive へのアップロード不要）。
コード更新時は再起動セルで `git pull` するだけで反映できます。

### B. 動画フォルダを Drive 上の好きな場所に作る

**動画を置く場所は完全に任意**。例えば:

```
MyDrive/
└── my-videos/                ← 動画フォルダ ← GUI で指定する
    ├── video-a.mp4
    ├── video-b.mkv
    └── converted/            ← CLI が自動生成（変換結果）
        └── video-a/
            ├── hls/master.m3u8 + *.ts
            ├── thumbs/poster.png + thumb_{05,30,50,60,80}.jpg
            └── meta.json
```

別の場所でも良い（`MyDrive/projects/lecture-2026/source/` など）。GUI のテキスト欄に絶対パスを入れて保存するだけ。

## 実行手順

1. **「セットアップ」セル**: FFmpeg / 依存 / Drive マウント / GitHub clone
2. **「サーバ起動」セル**: GUI を公開 URL で起動
3. **GUI 上で動画フォルダのパスを入力 → 保存**（例: `/content/drive/MyDrive/my-videos`）
4. **「変換 (CLI)」セル**: 動画を変換（GUI 設定のパスを自動参照）
5. GUI で「↻ 更新」して一覧を再読込

コード更新時は末尾の「コード更新時の再起動」セクション（`git pull`）を参照。


## 1. セットアップ (FFmpeg + Drive マウント + GitHub clone)

1 回実行するだけで以下を行います:

1. **FFmpeg + Python 依存** インストール、NVENC/CUVID 可用性の自己テスト
2. **Google Drive マウント** (`/content/drive`) — 動画フォルダへのアクセスに必要
3. **GitHub からアプリ本体を clone** (`/content/hls-video-player`)
4. **Python パッケージインストール** (`pip install -e .`)
5. **FFmpeg 関連の環境変数設定**

**動画フォルダはこのセルでは触りません**。サーバ起動後に GUI で指定してください。
（初期値として `/content/drive/MyDrive/` を入れておくので、GUI でサブフォルダ名を足すだけでOK）

### Private リポジトリの場合

リポジトリが private なら **Colab Secrets** に Personal Access Token を `GITHUB_TOKEN` として登録してください:

1. GitHub: Settings → Developer settings → Personal access tokens → Generate new token (classic)
2. スコープに `repo` (private リポジトリの read 権限) を付与
3. Colab 左サイドの 🔑 アイコン → Secrets → Add new → Name: `GITHUB_TOKEN`、Value: 生成した token
4. このノートブックへのアクセスを許可

Public リポジトリなら何もしなくて OK。


In [ ]:
# ============================================================
# 1. FFmpeg + Python 依存 + LD_LIBRARY_PATH 設定
# ============================================================
!apt-get -qq install -y ffmpeg git > /dev/null
!ffmpeg -version | head -1

# Colab T4 固有の NVIDIA driver パス対応
import os
_nvidia_lib = '/usr/lib64-nvidia'
if os.path.isdir(_nvidia_lib):
    cur = os.environ.get('LD_LIBRARY_PATH', '')
    if _nvidia_lib not in cur.split(':'):
        os.environ['LD_LIBRARY_PATH'] = _nvidia_lib + (':' + cur if cur else '')
print('LD_LIBRARY_PATH =', os.environ.get('LD_LIBRARY_PATH', ''))

!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E 'h264_nvenc|libx264' || true
!ffmpeg -hide_banner -decoders 2>/dev/null | grep -E 'h264_cuvid|hevc_cuvid|vp9_cuvid' || true
!nvidia-smi -L 2>/dev/null || echo '(GPU ランタイム未選択: CPU エンコードになります)'

print('--- CUDA device init self-test ---')
!ffmpeg -hide_banner -v error -f lavfi -i testsrc=d=0.1:s=320x240 -vf hwupload_cuda -f null - && echo 'CUVID/NVENC READY' || echo 'CUDA init FAILED (will fall back to CPU)'

!pip install -q gradio fastapi uvicorn python-multipart

# ============================================================
# 2. Google Drive マウント (動画フォルダへのアクセスに必要)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# 3. GitHub からアプリ本体を clone (既にあれば pull)
# ============================================================
GIT_REPO_URL = 'https://github.com/motowo/hls-video-player.git'
LOCAL_ROOT   = '/content/hls-video-player'

# Private リポジトリ用: Colab Secrets に GITHUB_TOKEN があれば URL に埋め込む。
# Public リポジトリならスキップされる (Secrets 未設定でも続行)。
try:
    from google.colab import userdata
    _token = userdata.get('GITHUB_TOKEN')
    if _token:
        GIT_REPO_URL = GIT_REPO_URL.replace('https://', f'https://{_token}@')
        print('✓ Colab Secrets の GITHUB_TOKEN を使って認証します')
except Exception:
    pass    # userdata 未対応 / Secret 未登録 — public 前提で続行

if os.path.isdir(f'{LOCAL_ROOT}/.git'):
    !cd {LOCAL_ROOT} && git pull
else:
    !rm -rf {LOCAL_ROOT}
    !git clone {GIT_REPO_URL} {LOCAL_ROOT}

%cd {LOCAL_ROOT}
!git log -1 --oneline

# ============================================================
# 4. Python パッケージとしてインストール
# ============================================================
!pip install -q -e .

# ============================================================
# 5. ライブラリパスの初期値（GUI のテキスト欄プリセット）
# ============================================================
# 動画フォルダは GUI で指定するが、毎回 Drive のルートまで打つのは面倒なので
# 「未設定なら Drive ルート」を初期値にしておく（既に設定済みなら尊重）。
from hls_video.library_settings import get_library_root, set_library_root
_DRIVE_HOME = '/content/drive/MyDrive'
_current = str(get_library_root())
if not os.path.isdir(_current) or _current.endswith('/library'):
    set_library_root(_DRIVE_HOME)
    print(f'library_root (初期値) = {_DRIVE_HOME}')
    print('  → GUI のテキスト欄でサブフォルダを指定してください (例: /content/drive/MyDrive/my-videos)')
else:
    print(f'library_root (前回値を保持) = {_current}')

# ============================================================
# 6. 変換パフォーマンス関連の環境変数
# ============================================================
os.environ['PORT'] = '7860'
os.environ['FFMPEG_HWACCEL'] = 'auto'
os.environ['FFMPEG_CUVID'] = 'auto'
os.environ['FFMPEG_NVENC_PRESET'] = 'p4'
os.environ['FFMPEG_BFRAMES'] = '0'
os.environ['FFMPEG_PRESET'] = 'ultrafast'
os.environ['FFMPEG_X264_TUNE'] = 'zerolatency'
os.environ['FFMPEG_AUDIO_COPY'] = '1'
os.environ['FFMPEG_THREADS'] = '0'
os.environ['FFMPEG_VARIANTS'] = '720p,360p'

print('\n環境変数設定完了')


## 2. サーバ起動 (FastAPI + Gradio)

FastAPI 本体 (port 7860) を `setup_tunnel` で直接トンネル化して `*.gradio.live` URL を発行します。

起動後の流れ:
1. 公開 URL を開く
2. **GUI 上部の「ライブラリのパス」テキスト欄**に動画フォルダの **絶対パス** を入力
   - 例: `/content/drive/MyDrive/my-videos`
3. 「保存して反映」ボタン（または Enter）で確定
4. 既に変換済みの動画があれば即座に一覧に表示される
5. 未変換動画は次のセル（変換 CLI）で処理


In [ ]:
# ============================================================
# FastAPI + Gradio 起動、*.gradio.live トンネル発行
# ============================================================
import gradio as gr
from fastapi import FastAPI

from app.gradio_library_ui import build_ui
from app.player_embed import router as player_router
from app.static_mount import mount_static

fapi = FastAPI()
mount_static(fapi, static_dir=f'{LOCAL_ROOT}/static')
fapi.include_router(player_router)

demo = build_ui()        # ライブラリパスは settings.json から自動解決
demo.queue()
gr.mount_gradio_app(fapi, demo, path='/')

import threading, time, uvicorn
_HLS_UVICORN = uvicorn.Server(uvicorn.Config(fapi, host='0.0.0.0', port=7860, log_level='warning'))
_HLS_THREAD = threading.Thread(target=_HLS_UVICORN.run, daemon=True)
_HLS_THREAD.start()
time.sleep(3)

import secrets
from gradio.networking import setup_tunnel

share_url = setup_tunnel(
    local_host='localhost',
    local_port=7860,
    share_token=secrets.token_urlsafe(32),
    share_server_address=None,
    share_server_tls_certificate=None,
)

print(f'\n🌐 公開 URL: {share_url}')
print('   /            → Gradio UI (パス設定 + 一覧 + 再生)')
print('   /play        → スマホ向け一覧')
print('   /player/<id> → プレイヤー単体ページ')
print('   /library/*   → HLS / サムネ / poster の動的配信')
print('   /api/videos  → 一覧 JSON API')
print('   /api/settings → 現在のライブラリパス')
print('\n💻 Colab 内からのアクセス: http://localhost:7860')
print('\n👉 公開 URL を開いたら、GUI のテキスト欄に動画フォルダのパスを入力してください。')
print('   例: /content/drive/MyDrive/my-videos')


## 3. 変換 (CLI)

GUI で指定した動画フォルダ配下の動画を一括変換します。

CLI のライブラリパス解決順:
1. コマンドライン引数: `!python -m hls_video.library_cli /content/drive/MyDrive/my-videos`
2. GUI 設定ファイル (`~/.config/hls-video-player/settings.json`)
3. 環境変数 `LIBRARY_ROOT`
4. `./library`（フォールバック）

**通常は引数なしで実行**して GUI で設定したパスを自動参照させればOK。

変換結果は **指定フォルダ直下の `converted/{stem}/`** に出力されます。
既に変換済みのもの (`converted/{stem}/hls/master.m3u8` 等が揃ってる) はスキップ（再変換は `--force`）。

オプション:
- `-w N` 並列ワーカー数（既定 2、GPU/CPU 余裕に応じて増減）
- `--filter <substr>` ファイル名フィルタ（部分一致）
- `--dry-run` 一覧のみ表示
- `-f` / `--force` 既存出力を再生成


In [ ]:
# ============================================================
# CLI で動画フォルダを一括変換 (HLS + 5 枚ダイジェスト + poster)
# ============================================================
# 動画を新しく追加した／再変換したいときだけこのセルを実行する。
# サーバ起動後の library_catalog は実行時にディスクを走査するので、
# 変換完了後に GUI の「↻ 更新」を押せば即反映される。

# 引数なし → settings.json (= GUI 設定) から自動解決
!python -m hls_video.library_cli -w 2

# 個別指定したい場合の例:
# !python -m hls_video.library_cli /content/drive/MyDrive/別フォルダ -w 4
# !python -m hls_video.library_cli --filter sample01 -w 1
# !python -m hls_video.library_cli --force -w 2     # 全部再生成
# !python -m hls_video.library_cli --dry-run        # 走査のみ


### 公開 URL のライフサイクル

- `*.gradio.live` のトンネルは **72 時間** で自動失効（Gradio 側の仕様）
- Colab ランタイム切断時にも失効する
- 再発行は上のセルを再実行するだけ（新しい URL が振られる）
- 長く稼働させ続けたい場合は Colab Pro + `Keep session active` や ngrok などの外部トンネルを検討

### トラブルシューティング

| 症状 | 対処 |
|---|---|
| 公開 URL を開くと 502 / 接続エラー | uvicorn が落ちた可能性。`!ss -tlnp \| grep 7860` で確認し、「サーバ起動」セルを再実行 |
| `setup_tunnel` が `TunnelException` で失敗 | Gradio 共有サーバ側の障害。数分待って再実行 |
| GUI に動画が出ない | (1) GUI のテキスト欄で正しいパスを設定したか / (2) CLI 変換が完了しているか (`!ls <path>/converted/`) |
| 「パスが存在しません」と言われる | Drive がマウントされているか確認 (`!ls /content/drive/MyDrive/`)。スペースを含むパスは GUI ではそのまま貼ればOK |
| ホバーしてもサムネが切り替わらない | `/library/<stem>/thumbs/thumb_05.jpg` が公開 URL 経由で 200 を返すか確認 |
| 変換が遅い / OOM | `-w 1` を使う、`FFMPEG_VARIANTS` を絞る、`FFMPEG_AUDIO_COPY=1` |
| 別の動画フォルダに切り替えたい | GUI のテキスト欄を書き換えて「保存して反映」だけでOK（再起動不要） |


## ログ確認 (リアルタイム)

変換や probe の進捗は別スレッドで動くため、サーバセルの実行が終わった後はそのセル出力には追加のログが流れない（Colab 仕様）。
`setup_logging()` は `/tmp/hls-video.log` にもファイル出力しているので、別セルで `!tail -f` すれば進捗・警告・エラーがリアルタイムに読める。

- 直近 200 行を眺めるだけ: 下の「ログ末尾」セル
- リアルタイムで追う: 下の「ログをストリーム表示」セル（停止はセルの停止ボタン）

In [ ]:
# ログ末尾（一発実行）
!tail -n 200 /tmp/hls-video.log 2>/dev/null || echo '(ログファイルがまだありません)'

In [ ]:
# ログをストリーム表示（止めるにはセルの停止ボタン ⬛）
!touch /tmp/hls-video.log && tail -n 50 -F /tmp/hls-video.log

## コード更新時の再起動

GitHub の最新コードに更新するときは、下のセルを 1 つ実行するだけ。

処理内容:
1. 既存の uvicorn サーバを停止 (`_HLS_UVICORN.should_exit = True`)
2. `frpc` トンネルプロセスを終了（新しい `*.gradio.live` URL を発行するため）
3. **`git pull`** で最新コードに更新（origin/main に hard reset）
4. `pip install -e .`（`pyproject.toml` 変更時のみ実質効く）
5. `sys.modules` から `app.*` / `hls_video.*` を破棄して再読込
6. FastAPI + Gradio を再起動、新しい `*.gradio.live` URL を表示

※ Drive マウントや FFmpeg 等の環境構築セルは触らない。Python kernel が
   生きていれば 10 秒ほどで再起動できる。

※ ローカルで編集したコードを Drive 経由で読ませる旧フローは廃止。
   コード変更は GitHub に push してから `git pull` で取得する流れになる。


In [ ]:
# --- LD_LIBRARY_PATH の再確認 (カーネル再接続対策) ---
import os
if '/usr/lib64-nvidia' not in os.environ.get('LD_LIBRARY_PATH', '').split(':'):
    os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')

# 1) 既存サーバ停止
try:
    _HLS_UVICORN.should_exit = True
    _HLS_THREAD.join(timeout=5)
    print('✓ uvicorn stopped')
except NameError:
    print('(前回起動なし)')
except Exception as e:
    print(f'stop error: {e}')

import subprocess, time
subprocess.run(['pkill', '-f', 'frpc'], check=False)
time.sleep(1)

# 2) GitHub から最新コードを pull
!cd {LOCAL_ROOT} && git pull
%cd {LOCAL_ROOT}
!git log -1 --oneline

# 3) 依存関係更新
!pip install -q -e .

# 4) sys.modules パージ
import sys
purged = [m for m in list(sys.modules) if m == 'app' or m == 'hls_video' or m.startswith(('app.', 'hls_video.'))]
for m in purged:
    sys.modules.pop(m, None)
print(f'✓ purged {len(purged)} modules')

# 5) FastAPI + Gradio を再起動
import gradio as gr
from fastapi import FastAPI
from app.gradio_library_ui import build_ui
from app.player_embed import router as player_router
from app.static_mount import mount_static

fapi = FastAPI()
mount_static(fapi, static_dir=f'{LOCAL_ROOT}/static')
fapi.include_router(player_router)

demo = build_ui()
demo.queue()
gr.mount_gradio_app(fapi, demo, path='/')

import threading, uvicorn
_HLS_UVICORN = uvicorn.Server(uvicorn.Config(fapi, host='0.0.0.0', port=7860, log_level='warning'))
_HLS_THREAD = threading.Thread(target=_HLS_UVICORN.run, daemon=True)
_HLS_THREAD.start()
time.sleep(3)

# 6) 新しい Gradio 共有トンネル発行
import secrets
from gradio.networking import setup_tunnel
share_url = setup_tunnel(
    local_host='localhost', local_port=7860,
    share_token=secrets.token_urlsafe(32),
    share_server_address=None, share_server_tls_certificate=None,
)
print(f'\n🔄 再起動完了 (前回 GUI で設定した動画フォルダパスは保持)')
print(f'🌐 新しい公開 URL: {share_url}')
